In [ ]:
# import json

In [1]:
__version__ = '0.7.2'

from typing import List, Optional

import torch
import torch.nn as nn


class CRF(nn.Module):
    """Conditional random field.

    This module implements a conditional random field [LMP01]_. The forward computation
    of this class computes the log likelihood of the given sequence of tags and
    emission score tensor. This class also has `~CRF.decode` method which finds
    the best tag sequence given an emission score tensor using `Viterbi algorithm`_.

    Args:
        num_tags: Number of tags.
        batch_first: Whether the first dimension corresponds to the size of a minibatch.

    Attributes:
        start_transitions (`~torch.nn.Parameter`): Start transition score tensor of size
            ``(num_tags,)``.
        end_transitions (`~torch.nn.Parameter`): End transition score tensor of size
            ``(num_tags,)``.
        transitions (`~torch.nn.Parameter`): Transition score tensor of size
            ``(num_tags, num_tags)``.


    .. [LMP01] Lafferty, J., McCallum, A., Pereira, F. (2001).
       "Conditional random fields: Probabilistic models for segmenting and
       labeling sequence data". *Proc. 18th International Conf. on Machine
       Learning*. Morgan Kaufmann. pp. 282–289.

    .. _Viterbi algorithm: https://en.wikipedia.org/wiki/Viterbi_algorithm
    """

    def __init__(self, num_tags: int, batch_first: bool = False) -> None:
        if num_tags <= 0:
            raise ValueError(f'invalid number of tags: {num_tags}')
        super().__init__()
        self.num_tags = num_tags
        self.batch_first = batch_first
        self.start_transitions = nn.Parameter(torch.empty(num_tags))
        self.end_transitions = nn.Parameter(torch.empty(num_tags))
        self.transitions = nn.Parameter(torch.empty(num_tags, num_tags))

        self.reset_parameters()

    def reset_parameters(self) -> None:
        """Initialize the transition parameters.

        The parameters will be initialized randomly from a uniform distribution
        between -0.1 and 0.1.
        """
        nn.init.uniform_(self.start_transitions, -0.1, 0.1)
        nn.init.uniform_(self.end_transitions, -0.1, 0.1)
        nn.init.uniform_(self.transitions, -0.1, 0.1)

    def __repr__(self) -> str:
        return f'{self.__class__.__name__}(num_tags={self.num_tags})'

    def forward(
            self,
            emissions: torch.Tensor,
            tags: torch.LongTensor,
            mask: Optional[torch.ByteTensor] = None,
            reduction: str = 'sum',
    ) -> torch.Tensor:
        """Compute the conditional log likelihood of a sequence of tags given emission scores.

        Args:
            emissions (`~torch.Tensor`): Emission score tensor of size
                ``(seq_length, batch_size, num_tags)`` if ``batch_first`` is ``False``,
                ``(batch_size, seq_length, num_tags)`` otherwise.
            tags (`~torch.LongTensor`): Sequence of tags tensor of size
                ``(seq_length, batch_size)`` if ``batch_first`` is ``False``,
                ``(batch_size, seq_length)`` otherwise.
            mask (`~torch.ByteTensor`): Mask tensor of size ``(seq_length, batch_size)``
                if ``batch_first`` is ``False``, ``(batch_size, seq_length)`` otherwise.
            reduction: Specifies  the reduction to apply to the output:
                ``none|sum|mean|token_mean``. ``none``: no reduction will be applied.
                ``sum``: the output will be summed over batches. ``mean``: the output will be
                averaged over batches. ``token_mean``: the output will be averaged over tokens.

        Returns:
            `~torch.Tensor`: The log likelihood. This will have size ``(batch_size,)`` if
            reduction is ``none``, ``()`` otherwise.
        """
        self._validate(emissions, tags=tags, mask=mask)
        if reduction not in ('none', 'sum', 'mean', 'token_mean'):
            raise ValueError(f'invalid reduction: {reduction}')
        if mask is None:
            mask = torch.ones_like(tags, dtype=torch.uint8)

        if self.batch_first:
            emissions = emissions.transpose(0, 1)
            tags = tags.transpose(0, 1)
            mask = mask.transpose(0, 1)

        # shape: (batch_size,)
        numerator = self._compute_score(emissions, tags, mask)
        # shape: (batch_size,)
        denominator = self._compute_normalizer(emissions, mask)
        # shape: (batch_size,)
        llh = numerator - denominator

        if reduction == 'none':
            return llh
        if reduction == 'sum':
            return llh.sum()
        if reduction == 'mean':
            return llh.mean()
        assert reduction == 'token_mean'
        return llh.sum() / mask.type_as(emissions).sum()

    @torch.jit.export
    def decode(self, emissions: torch.Tensor,
               mask: Optional[torch.ByteTensor] = None) -> List[List[int]]:
        """Find the most likely tag sequence using Viterbi algorithm.

        Args:
            emissions (`~torch.Tensor`): Emission score tensor of size
                ``(seq_length, batch_size, num_tags)`` if ``batch_first`` is ``False``,
                ``(batch_size, seq_length, num_tags)`` otherwise.
            mask (`~torch.ByteTensor`): Mask tensor of size ``(seq_length, batch_size)``
                if ``batch_first`` is ``False``, ``(batch_size, seq_length)`` otherwise.

        Returns:
            List of list containing the best tag sequence for each batch.
        """
        self._validate(emissions, mask=mask)
        if mask is None:
            mask = emissions.new_ones(emissions.shape[:2], dtype=torch.uint8)

        if self.batch_first:
            emissions = emissions.transpose(0, 1)
            mask = mask.transpose(0, 1)

        return self._viterbi_decode(emissions, mask)

    def _validate(
            self,
            emissions: torch.Tensor,
            tags: Optional[torch.LongTensor] = None,
            mask: Optional[torch.ByteTensor] = None) -> None:
        if emissions.dim() != 3:
            raise ValueError(f'emissions must have dimension of 3, got {emissions.dim()}')
        if emissions.size(2) != self.num_tags:
            raise ValueError(
                f'expected last dimension of emissions is {self.num_tags}, '
                f'got {emissions.size(2)}')

        if tags is not None:
            if emissions.shape[:2] != tags.shape:
                raise ValueError(
                    'the first two dimensions of emissions and tags must match, '
                    f'got {(emissions.shape[0], emissions.shape[1])} and {(tags.shape[0], tags.shape[1])}'
                )

        if mask is not None:
            if emissions.shape[:2] != mask.shape:
                raise ValueError(
                    'the first two dimensions of emissions and mask must match, '
                    f'got {(emissions.shape[0], emissions.shape[1])} and {(mask.shape[0], mask.shape[1])}'
                )
            no_empty_seq = not self.batch_first and mask[0].all()
            no_empty_seq_bf = self.batch_first and mask[:, 0].all()
            if not no_empty_seq and not no_empty_seq_bf:
                raise ValueError('mask of the first timestep must all be on')

    def _compute_score(
            self, emissions: torch.Tensor, tags: torch.LongTensor,
            mask: torch.ByteTensor) -> torch.Tensor:
        # emissions: (seq_length, batch_size, num_tags)
        # tags: (seq_length, batch_size)
        # mask: (seq_length, batch_size)
        assert emissions.dim() == 3 and tags.dim() == 2
        assert emissions.shape[:2] == tags.shape
        assert emissions.size(2) == self.num_tags
        assert mask.shape == tags.shape
        assert mask[0].all()

        seq_length, batch_size = tags.shape
        mask = mask.type_as(emissions)

        # Start transition score and first emission
        # shape: (batch_size,)
        score = self.start_transitions[tags[0]]
        score += emissions[0, torch.arange(batch_size), tags[0]]

        for i in range(1, seq_length):
            # Transition score to next tag, only added if next timestep is valid (mask == 1)
            # shape: (batch_size,)
            score += self.transitions[tags[i - 1], tags[i]] * mask[i]

            # Emission score for next tag, only added if next timestep is valid (mask == 1)
            # shape: (batch_size,)
            score += emissions[i, torch.arange(batch_size), tags[i]] * mask[i]

        # End transition score
        # shape: (batch_size,)
        seq_ends = mask.long().sum(dim=0) - 1
        # shape: (batch_size,)
        last_tags = tags[seq_ends, torch.arange(batch_size)]
        # shape: (batch_size,)
        score += self.end_transitions[last_tags]

        return score

    def _compute_normalizer(
            self, emissions: torch.Tensor, mask: torch.ByteTensor) -> torch.Tensor:
        # emissions: (seq_length, batch_size, num_tags)
        # mask: (seq_length, batch_size)
        assert emissions.dim() == 3 and mask.dim() == 2
        assert emissions.shape[:2] == mask.shape
        assert emissions.size(2) == self.num_tags
        assert mask[0].all()

        seq_length = emissions.size(0)

        # Start transition score and first emission; score has size of
        # (batch_size, num_tags) where for each batch, the j-th column stores
        # the score that the first timestep has tag j
        # shape: (batch_size, num_tags)
        score = self.start_transitions + emissions[0]

        for i in range(1, seq_length):
            # Broadcast score for every possible next tag
            # shape: (batch_size, num_tags, 1)
            broadcast_score = score.unsqueeze(2)

            # Broadcast emission score for every possible current tag
            # shape: (batch_size, 1, num_tags)
            broadcast_emissions = emissions[i].unsqueeze(1)

            # Compute the score tensor of size (batch_size, num_tags, num_tags) where
            # for each sample, entry at row i and column j stores the sum of scores of all
            # possible tag sequences so far that end with transitioning from tag i to tag j
            # and emitting
            # shape: (batch_size, num_tags, num_tags)
            next_score = broadcast_score + self.transitions + broadcast_emissions

            # Sum over all possible current tags, but we're in score space, so a sum
            # becomes a log-sum-exp: for each sample, entry i stores the sum of scores of
            # all possible tag sequences so far, that end in tag i
            # shape: (batch_size, num_tags)
            next_score = torch.logsumexp(next_score, dim=1)

            # Set score to the next score if this timestep is valid (mask == 1)
            # shape: (batch_size, num_tags)
            score = torch.where(mask[i].unsqueeze(1), next_score, score)

        # End transition score
        # shape: (batch_size, num_tags)
        score += self.end_transitions

        # Sum (log-sum-exp) over all possible tags
        # shape: (batch_size,)
        return torch.logsumexp(score, dim=1)

    def _viterbi_decode(self, emissions: torch.FloatTensor,
                        mask: torch.ByteTensor) -> List[List[int]]:
        # emissions: (seq_length, batch_size, num_tags)
        # mask: (seq_length, batch_size)
        assert emissions.dim() == 3 and mask.dim() == 2
        assert emissions.shape[:2] == mask.shape
        assert emissions.size(2) == self.num_tags
        assert mask[0].all()

        seq_length, batch_size = mask.shape

        # Start transition and first emission
        # shape: (batch_size, num_tags)
        score = self.start_transitions + emissions[0]
        history: List[torch.Tensor] = []

        # score is a tensor of size (batch_size, num_tags) where for every batch,
        # value at column j stores the score of the best tag sequence so far that ends
        # with tag j
        # history saves where the best tags candidate transitioned from; this is used
        # when we trace back the best tag sequence

        # Viterbi algorithm recursive case: we compute the score of the best tag sequence
        # for every possible next tag
        for i in range(1, seq_length):
            # Broadcast viterbi score for every possible next tag
            # shape: (batch_size, num_tags, 1)
            broadcast_score = score.unsqueeze(2)

            # Broadcast emission score for every possible current tag
            # shape: (batch_size, 1, num_tags)
            broadcast_emission = emissions[i].unsqueeze(1)

            # Compute the score tensor of size (batch_size, num_tags, num_tags) where
            # for each sample, entry at row i and column j stores the score of the best
            # tag sequence so far that ends with transitioning from tag i to tag j and emitting
            # shape: (batch_size, num_tags, num_tags)
            next_score = broadcast_score + self.transitions + broadcast_emission

            # Find the maximum score over all possible current tag
            # shape: (batch_size, num_tags)
            next_score, indices = next_score.max(dim=1)

            # Set score to the next score if this timestep is valid (mask == 1)
            # and save the index that produces the next score
            # shape: (batch_size, num_tags)
            score = torch.where(mask[i].unsqueeze(1), next_score, score)
            history.append(indices)

        # End transition score
        # shape: (batch_size, num_tags)
        score += self.end_transitions

        # Now, compute the best path for each sample

        # shape: (batch_size,)
        seq_ends = mask.long().sum(dim=0) - 1
        best_tags_list: List[List[int]] = []

        for idx in range(batch_size):
            # Find the tag which maximizes the score at the last timestep; this is our best tag
            # for the last timestep
            _, best_last_tag = score[idx].max(dim=0)
            best_tags: List[int] = []
            best_tags.append(best_last_tag.item())

            # We trace back where the best last tag comes from, append that to our best tag
            # sequence, and trace it back again, and so on
            # NOTE: reversed() cannot be used here because it is not supported by TorchScript,
            # see https://github.com/pytorch/pytorch/issues/31772.
            for hist in history[:seq_ends[idx]][::-1]:
                best_last_tag = hist[idx][best_tags[-1]]
                best_tags.append(best_last_tag.item())

            # Reverse the order because we start from the last timestep
            best_tags.reverse()
            best_tags_list.append(best_tags)

        return best_tags_list

In [4]:
import json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoConfig

In [5]:
class NERDataset(Dataset):
    def __init__(self, data_path, tokenizer, label_pad_id=-100, max_length=128):
        with open(data_path, "r", encoding="utf-8") as f:
            raw = json.load(f)["examples"]
        self.data = raw
        self.tokenizer = tokenizer
        self.label_pad_id = label_pad_id
        self.max_length = max_length

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        tokens = self.data[idx]["tokens"]
        ner_tags = self.data[idx]["ner_tags"]

        # buat encoding untuk tokens 
        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors="pt"
        )

        # align labels dengan tokens yang sudah diencoding (jadi kepotong2 sesuai tokenization)
        word_ids = encoding.word_ids(batch_index=0)
        aligned_labels = []
        previous_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                aligned_labels.append(self.label_pad_id)
            elif word_idx != previous_word_idx:
                aligned_labels.append(ner_tags[word_idx])
            else:
                aligned_labels.append(self.label_pad_id)
            previous_word_idx = word_idx
        
        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(aligned_labels, dtype=torch.long)
        }

        return item

In [6]:
def load_label_info(model_name):
    config = AutoConfig.from_pretrained(model_name)
    id2label = config.id2label
    label2id = config.label2id
    num_labels = config.num_labels

    label_info = {
        "id2label": id2label,
        "label2id": label2id,
        "num_labels": num_labels
    }

    return label_info

def create_dataloaders(
        train_path, val_path, test_path,
        model_name,
        batch_size=32,
        max_length=128
):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    train_dataset = NERDataset(train_path, tokenizer, max_length=max_length)
    val_dataset = NERDataset(val_path, tokenizer, max_length=max_length)
    test_dataset = NERDataset(test_path, tokenizer, max_length=max_length)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

In [7]:
train_loader, val_loader, test_loader = create_dataloaders(
    train_path=r"D:\Dafa\Project\queryner-kd\data\processed\train.json",
    val_path=r"D:\Dafa\Project\queryner-kd\data\processed\validation.json",
    test_path=r"D:\Dafa\Project\queryner-kd\data\processed\test.json",
    model_name="bert-base-uncased",
    batch_size=16,
    max_length=128
)

label_info = load_label_info("bltlab/queryner-augmented-data-bert-base-uncased")

In [13]:
batch = next(iter(train_loader))
batch.keys()

dict_keys(['input_ids', 'attention_mask', 'labels'])

In [10]:
from torch import nn
from torchcrf import CRF
from transformers import AutoModel, AutoConfig

In [11]:
# BaseNERModel
label_info = label_info
use_crf = False

# QueryNER
model_name = "bert-base-uncased"
config = AutoConfig.from_pretrained(
    model_name,
    num_labels=label_info["num_labels"],
    id2label=label_info["id2label"],
    label2id=label_info["label2id"]
)

bert = AutoModel.from_pretrained(model_name, config=config)
dropout = nn.Dropout(0.3)
classifier = nn.Linear(bert.config.hidden_size, bert.config.num_labels)
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

In [14]:
# forward
outputs = bert(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
outputs.last_hidden_state.shape  # (batch_size, seq_len, hidden_size)

torch.Size([16, 128, 768])

In [15]:
sequence_output = dropout(outputs.last_hidden_state)
sequence_output.shape

torch.Size([16, 128, 768])

#### Use CRF False

awalnya dari dim 2 nya num hidden layer jadi num_labels

In [ ]:
logits = classifier(sequence_output)  # (batch_size, seq_len, num_labels)
logits.shape

torch.Size([16, 128, 35])

In [ ]:
loss = loss_fn(
    logits.view(-1, label_info["num_labels"]),
    batch["labels"].view(-1)
)
loss

tensor(3.6103, grad_fn=<NllLossBackward0>)

#### Use CRF True

In [16]:
from torchcrf import CRF
fc = nn.Linear(bert.config.hidden_size, bert.config.num_labels)
crf = CRF(label_info["num_labels"], batch_first=True)

In [163]:
emissions = fc(sequence_output)  # (batch_size, seq_len, num_labels)
emissions.shape

torch.Size([16, 128, 35])

In [164]:
fc(sequence_output)[0][0]

tensor([ 0.1516,  0.1849, -0.2573,  0.5031,  0.0028, -0.1282, -0.0375,  0.2492,
         0.0283, -0.0381, -0.0768, -0.0874, -0.4471,  0.3482, -0.1081,  0.0351,
         0.1090, -0.4375, -0.1550, -0.3741, -0.2502,  0.8278,  0.5491, -0.1636,
         0.4661, -0.2638,  0.8029, -0.1700,  0.0982,  0.0108, -0.1312,  0.3523,
         0.3958, -0.1178, -0.3155], grad_fn=<SelectBackward0>)

In [165]:
tags = batch["labels"].clone()
tags[tags == -100] = 0
tags.shape

torch.Size([16, 128])

In [166]:
mask = batch["attention_mask"]
mask = mask.bool()
mask[:, 0] = True
mask.shape

torch.Size([16, 128])

### Log likehood loss

In [29]:
# log_likehood = crf(emissions, batch["labels"], mask=batch["attention_mask"].bool(), reduction='token_mean')
# loss = -log_likehood
# loss

### CRF decode

In [30]:
# pred = crf.decode(emissions, mask=batch["attention_mask"].bool())
# pred

# CObain buat crf sendiri

In [140]:
import torch
import torch.nn as nn

In [141]:
num_tags = label_info["num_labels"]

In [47]:
start_transitions = nn.Parameter(torch.empty(num_tags))
end_transitions = nn.Parameter(torch.empty(num_tags))
transitions = nn.Parameter(torch.empty(num_tags, num_tags))
start_transitions

Parameter containing:
tensor([ 1.4070e+11,  9.5569e-43, -2.5733e-01,  3.9585e-01, -1.1779e-01,
        -3.1546e-01,  4.6176e-01, -3.2746e-01, -5.7094e-01,  8.6864e-01,
         8.9114e-02, -5.0074e-01,  5.2537e-01,  4.8893e-02,  1.6429e-01,
         5.8760e-01, -1.5936e-01, -3.7770e-01,  4.0074e-01, -3.0967e-01,
         2.8394e-01,  8.5773e-01, -1.9372e-01, -8.7018e-01,  7.8662e-01,
        -4.9241e-01,  3.8452e-02,  4.4658e-01, -2.2592e-01, -4.3872e-01,
         8.6132e-01, -1.6293e-01, -8.0786e-02,  7.8436e-01, -1.4578e-01],
       requires_grad=True)

In [48]:
nn.init.uniform_(start_transitions, -0.1, 0.1)
nn.init.uniform_(end_transitions, -0.1, 0.1)
nn.init.uniform_(transitions, -0.1, 0.1)
start_transitions

Parameter containing:
tensor([ 0.0314,  0.0788,  0.0930, -0.0613, -0.0751,  0.0117, -0.0060,  0.0366,
        -0.0034, -0.0347, -0.0433,  0.0635,  0.0558,  0.0987, -0.0157,  0.0352,
        -0.0022, -0.0245, -0.0741,  0.0155,  0.0817,  0.0529,  0.0655, -0.0722,
        -0.0473, -0.0816, -0.0850,  0.0473, -0.0100, -0.0674, -0.0452, -0.0296,
         0.0808, -0.0072, -0.0294], requires_grad=True)

In [167]:
emissions = emissions.transpose(0, 1)
tags = tags.transpose(0, 1)
mask = mask.transpose(0, 1)

In [168]:
print(emissions.shape, tags.shape, mask.shape)

torch.Size([128, 16, 35]) torch.Size([128, 16]) torch.Size([128, 16])


### numerator (_compute_score)

In [41]:
seq_length, batch_size = tags.shape
print(seq_length, batch_size)

128 16


In [46]:
# samain tiep datanya jadi sama dengan emissions
mask = mask.type_as(emissions)
mask.shape

torch.Size([128, 16])

In [55]:
tags[0]

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [53]:
start_transitions[tags[0]]

tensor([0.0314, 0.0314, 0.0314, 0.0314, 0.0314, 0.0314, 0.0314, 0.0314, 0.0314,
        0.0314, 0.0314, 0.0314, 0.0314, 0.0314, 0.0314, 0.0314],
       grad_fn=<IndexBackward0>)

In [54]:
score = start_transitions[tags[0]]

In [57]:
score += emissions[0, torch.arange(batch_size), tags[0]]

In [73]:
score

tensor([0.1830, 0.4932, 0.5568, 0.0650, 0.7447, 0.2346, 0.4652, 0.7276, 0.4832,
        0.6478, 0.5149, 0.2576, 0.6983, 0.4321, 0.8180, 0.8927],
       grad_fn=<AddBackward0>)

In [87]:
for i in range(1, seq_length):
    score += transitions[tags[i-1], tags[i]] * mask[i]
    score += emissions[i, torch.arange(batch_size), tags[i]] * mask[i]

In [88]:
score

tensor([0.4600, 0.7048, 0.8976, 0.3016, 1.4942, 0.1312, 0.6637, 1.0004, 2.0584,
        1.7961, 1.7349, 1.0083, 2.1077, 2.3965, 2.3353, 1.0067],
       grad_fn=<AddBackward0>)

In [90]:
seq_ends = mask.long().sum(dim=0) - 1
seq_ends

tensor([7, 5, 5, 4, 3, 4, 4, 8, 6, 8, 5, 7, 7, 7, 6, 3])

In [91]:
last_tags = tags[seq_ends, torch.arange(batch_size)]
last_tags

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [92]:
score += end_transitions[last_tags]
score

tensor([0.4387, 0.6835, 0.8763, 0.2803, 1.4729, 0.1099, 0.6424, 0.9791, 2.0371,
        1.7748, 1.7136, 0.9870, 2.0864, 2.3752, 2.3140, 0.9854],
       grad_fn=<AddBackward0>)

In [93]:
numerator_score = score

### normalizer

In [169]:
assert emissions.dim() == 3 and mask.dim() == 2
assert emissions.shape[:2] == mask.shape
assert emissions.size(2) == num_tags
assert mask[0].all()

In [170]:
seq_length = emissions.size(0)
seq_length

128

In [171]:
score = start_transitions + emissions[0]
score.shape

torch.Size([16, 35])

In [172]:
score.unsqueeze(2).shape

torch.Size([16, 35, 1])

In [173]:
emissions[1].unsqueeze(1).shape

torch.Size([16, 1, 35])

In [174]:
transitions.shape

torch.Size([35, 35])

In [175]:
(score.unsqueeze(2) + transitions + emissions[1].unsqueeze(1)).shape

torch.Size([16, 35, 35])

In [180]:
torch.logsumexp((score.unsqueeze(2) + transitions + emissions[1].unsqueeze(1)), dim=1).shape

torch.Size([16, 35])

In [178]:
for i in range(1, seq_length):
    broadcast_score = score.unsqueeze(2)
    broadcast_emissions = emissions[i].unsqueeze(1)
    next_score = broadcast_score + transitions + broadcast_emissions
    next_score = torch.logsumexp(next_score, dim=1)
    score = torch.where(mask[i].unsqueeze(1), next_score, score)

In [181]:
score += end_transitions
score.shape

torch.Size([16, 35])

In [183]:
denominator_score = torch.logsumexp(score, dim=1)

### LLH

In [184]:
llh = numerator_score - denominator_score
llh

tensor([-28.5457, -21.3927, -21.1739, -17.7134, -13.2704, -18.2306, -17.6635,
        -32.2304, -23.7024, -31.2885, -20.3612, -28.6100, -27.4723, -27.0961,
        -23.5526, -13.7525], grad_fn=<SubBackward0>)

In [186]:
mask.type_as(emissions).sum()

tensor(105.)

In [185]:
llh.sum() / mask.type_as(emissions).sum()

tensor(-3.4862, grad_fn=<DivBackward0>)

In [188]:
109511218-109509923

1295

In [191]:
start_transitions.shape

torch.Size([35])

In [193]:
print("The additional parameters are:")
print((35*35)+2*(35))

The additional parameters are:
1295


### Viterbi decoding

In [196]:
seq_length, batch_size = mask.shape

In [199]:
score = start_transitions + emissions[0]
score.shape

torch.Size([16, 35])

In [209]:
history = []

In [210]:
for i in range(1, seq_length):
    broadcast_score = score.unsqueeze(2)
    broadcast_emissions = emissions[i].unsqueeze(1)
    next_score = broadcast_score + transitions + broadcast_emissions
    next_score, indices = next_score.max(dim=1)
    score = torch.where(mask[i].unsqueeze(1), next_score, score)
    history.append(indices)

In [218]:
history[0][0]

tensor([26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26,
        26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26])

In [220]:
score += end_transitions
score.shape

torch.Size([16, 35])

In [221]:
mask.long().sum(dim=0) - 1

tensor([7, 5, 5, 4, 3, 4, 4, 8, 6, 8, 5, 7, 7, 7, 6, 3])

In [226]:
seq_ends = mask.long().sum(dim=0) - 1
best_tags_list: List[List[int]] = []
seq_ends

tensor([7, 5, 5, 4, 3, 4, 4, 8, 6, 8, 5, 7, 7, 7, 6, 3])

In [227]:
for idx in range(batch_size):
    _, best_last_tag = score[idx].max(dim=0)
    best_tags: List[int] = []
    best_tags.append(best_last_tag.item())

    for hist in history[:seq_ends[idx]][::-1]:
        best_last_tag = hist[idx][best_tags[-1]]
        best_tags.append(best_last_tag.item())
    
    best_tags.reverse()
    best_tags_list.append(best_tags)

In [228]:
best_tags_list

[[26, 26, 21, 1, 0, 27, 26, 31],
 [17, 19, 3, 4, 26, 31],
 [0, 0, 3, 4, 12, 20],
 [24, 24, 2, 7, 31],
 [24, 24, 26, 24],
 [18, 8, 3, 1, 24],
 [32, 32, 25, 3, 20],
 [7, 7, 7, 7, 8, 25, 9, 25, 31],
 [30, 33, 20, 0, 12, 26, 24],
 [0, 0, 22, 23, 30, 29, 5, 22, 24],
 [32, 32, 8, 5, 18, 6],
 [32, 5, 25, 11, 10, 0, 26, 24],
 [32, 5, 17, 11, 5, 0, 23, 16],
 [32, 32, 7, 31, 25, 31, 21, 24],
 [0, 0, 0, 0, 12, 26, 24],
 [11, 3, 18, 24]]

### diff between bruteforce vs viterbi

In [ ]:
# 35**8

2251875390625

In [ ]:
# 8*(35**2)

9800